# 06 — Model Training & Comparison

In this notebook, we train and evaluate multiple machine learning classifiers for employee attrition:
1. **Logistic Regression** (Linear baseline, handles class imbalance via `class_weight='balanced'`)
2. **Decision Tree Classifier** (Interpretable tree baseline)
3. **Random Forest Classifier** (Bagging ensemble)
4. **Gradient Boosting Classifier** (Boosting ensemble)

We then evaluate them on the test set using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib

data_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
models_dir = '../models' if os.path.exists('../models') else 'models'

X_train = pd.read_csv(f'{data_dir}/X_train.csv')
X_test = pd.read_csv(f'{data_dir}/X_test.csv')
y_train = pd.read_csv(f'{data_dir}/y_train.csv').values.ravel()
y_test = pd.read_csv(f'{data_dir}/y_test.csv').values.ravel()

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (1176, 44) Test shape: (294, 44)


## 1. Define Machine Learning Models

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Note: We use class_weight='balanced' where available to handle class imbalance (~16% attrition)
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

## 2. Train and Evaluate Models

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Model': name,
        'Accuracy': round(acc, 3),
        'Precision': round(prec, 3),
        'Recall': round(rec, 3),
        'F1-Score': round(f1, 3),
        'ROC-AUC': round(auc, 3)
    })

results_df = pd.DataFrame(results)
print("=== Model Performance Comparison ===")
display(results_df)

=== Model Performance Comparison ===


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.755,0.356,0.660,0.463,0.804
1,Decision Tree,0.772,0.357,0.532,0.427,0.652
2,Random Forest,0.837,0.444,0.085,0.143,0.759
3,Gradient Boosting,0.850,0.588,0.213,0.312,0.794


## 3. Save Trained Models & Best Model

In HR analytics, catching employees at risk of leaving is critical (high **Recall** and **ROC-AUC**).
- **Logistic Regression** achieves ~0.62 Recall and ~0.80 ROC-AUC with strong explainability.
- **Random Forest** provides strong non-linear interactions and feature importance rankings.

We save the best models and the comparison summary for deployment and our Streamlit application.

In [4]:
# Save comparison table
results_df.to_csv(f'{models_dir}/model_comparison.csv', index=False)

# Train Logistic Regression and Random Forest models
best_lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
best_lr.fit(X_train, y_train)

best_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
best_rf.fit(X_train, y_train)

prep_bundle = joblib.load(f'{models_dir}/preprocessor.pkl')

model_artifact = {
    'lr_model': best_lr,
    'rf_model': best_rf,
    'preprocessor': prep_bundle['preprocessor'],
    'feature_names': prep_bundle['feature_names'],
    'num_cols': prep_bundle['num_cols'],
    'cat_cols': prep_bundle['cat_cols'],
    'defaults': prep_bundle['defaults'],
    'comparison_df': results_df
}

joblib.dump(model_artifact, f'{models_dir}/attrition_model.pkl')
print("Model training complete! Artifact saved to models/attrition_model.pkl")

Model training complete! Artifact saved to models/attrition_model.pkl
